# 02 · Greenness: crop calendar, season dates and stress years

**Field Shift** (NASA Space Apps 2026), data notebook 2 of 6. *Code written with AI assistance (Claude, Anthropic); run and checked by the team.* Needs notebook 01's files in the same Drive folder.

| Output | What's in it |
|---|---|
| `data/field.json` (adds `greenness`) | The demo field: MODIS greenness every 16 days since 2000, every clear 30 m HLS view and 10 m Sentinel-2 view (with red-edge NDRE) since 2016, nearby rain-fed fields for comparison, and each season's peak, green-up and dry-off |
| `data/cells_greenness.json` | For every farmland square: typical greenness for each month (low / typical / high), typical season dates, bare months, and each season's peak and dry-off since 2001 |

**Greenness (NDVI)** is how green the ground looks from space: about 0.2 is bare soil, 0.8 is a thick healthy crop. A "season" is named by its harvest year: season 2024 runs from September 2023 to August 2024.

**How to run (about 10–15 minutes the first time):** put your project ID below, then **Runtime → Run all**. Everything is saved to Drive as it goes, so if Colab stops, run all again and it carries on.

In [ ]:
EE_PROJECT = "skilled-clover-514221-p6"    # Earth Engine Cloud project (change it if someone else runs this)
DRIVE_FOLDER = "field-shift"            # folder for this notebook's files (in Google Drive if USE_DRIVE is True)
USE_DRIVE = False                       # False: you pick the input files with an upload button. True: files are shared through Google Drive
REPO = ""                               # optional, e.g. "team-owner/field-shift": used only if a file isn't in Drive

## Setup

In [ ]:
import sys, subprocess
try:
    import pyproj, shapely                  # normally already in Colab
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyproj", "shapely"], check=True)

import json, math, re, time, random, hashlib, urllib.request, datetime as dt
import concurrent.futures as cf
from pathlib import Path

import ee
import numpy as np
import pandas as pd
from pyproj import Transformer
from shapely.geometry import shape

if EE_PROJECT == "your-cloud-project-id":
    raise ValueError("Put your Earth Engine Cloud project ID into EE_PROJECT (first code cell), then run again.")
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
print("Earth Engine ready")

IN_COLAB = "google.colab" in sys.modules
BASE = Path(DRIVE_FOLDER)
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    try:
        drive.mount("/content/drive")
    except Exception as e:
        raise RuntimeError(
            f"Google Drive didn't connect ({str(e)[:80]}).\n"
            "Run this cell again. In the Google window, pick the same account and tick 'Select all' "
            "(the Google Drive box) before pressing Continue.\n"
            "If Drive can't be used at all, set USE_DRIVE = False in the first cell. The notebook will then ask you to "
            "upload notebook 01's files, and you upload its downloads to the repo yourself.") from None
    BASE = Path("/content/drive/MyDrive") / DRIVE_FOLDER
elif IN_COLAB:
    BASE = Path("/content") / DRIVE_FOLDER
DATA, CACHE = BASE / "data", BASE / "cache"
for folder in (DATA, CACHE):
    folder.mkdir(parents=True, exist_ok=True)
print("Working folder:", BASE)

TO_LAEA = Transformer.from_crs("EPSG:4326", "EPSG:3035", always_xy=True)
CELL = 1000
RUN_AT = dt.datetime.now(dt.timezone.utc).strftime("%Y-%m-%dT%H:%MZ")
BASELINE = (2001, 2020)          # the "normal" period: at least 15 years, printed in every file

def with_retries(fn, tries=6):
    """Earth Engine sometimes answers 'Too many requests' or times out: wait, then try again."""
    for k in range(tries):
        try:
            return fn()
        except Exception as e:
            if k == tries - 1:
                raise
            print("  retrying after:", str(e)[:100])
            time.sleep(min(60, 2 ** k + random.random()))

CHECKS = []
def check(ok, text, warn=False, fail=None):
    """Record and print a check: ✅ passed, ⚠️ worth a look, ❌ needs fixing (fail = what to print if it didn't pass)."""
    mark = "✅" if ok else ("⚠️" if warn else "❌")
    line = f"{mark} {text if ok or not fail else fail}"
    CHECKS.append(line)
    print(line)

def num(v, nd):
    """Round a value for the JSON files; empty or missing values become null."""
    try:
        v = float(v)
    except (TypeError, ValueError):
        return None
    if math.isnan(v) or math.isinf(v):
        return None
    return int(round(v)) if nd == 0 else round(v, nd)

def lth(values, nd, min_n=15):
    """[low, typical, high] = 10th percentile, median, 90th percentile; null if fewer than min_n values."""
    v = pd.Series(values, dtype="float64").dropna()
    if len(v) < min_n:
        return [None, None, None]
    return [num(v.quantile(0.1), nd), num(v.median(), nd), num(v.quantile(0.9), nd)]

INPUTS = ["grid.geojson", "field.json"] + list(globals().get("OPTIONAL_INPUTS", []))
missing = [n for n in ["grid.geojson", "field.json"] if not (DATA / n).exists()]
if missing and IN_COLAB and not REPO:
    from google.colab import files
    print("Choose these files (downloaded from earlier notebooks) in the upload box below, all at once:")
    print("   " + ", ".join(INPUTS) + ("   (the last ones only if you have them)" if len(INPUTS) > 2 else ""))
    for fname, content in files.upload().items():
        clean = re.sub(r" \(\d+\)(?=\.)", "", Path(fname).name)        # "field (1).json" -> "field.json"
        (DATA / clean).write_bytes(content)
        print("   got", clean)

def load_optional(name):
    """A data file if it's in the working folder, else an empty dict (no questions asked)."""
    path = DATA / name
    return json.loads(path.read_text()) if path.exists() else {}

def write_field_section(part, sections, sources):
    """This notebook's part of the demo field goes in its own file, field_<part>.json; notebook 06 merges them into field.json."""
    doc = {"meta": {"project": "Field Shift (NASA Space Apps Challenge 2026)", "made_by": f"{NOTEBOOK} ({RUN_AT})",
                    "field_key": FIELD_KEY, "sources": sources}}
    doc.update(sections)
    return write_json(DATA / f"field_{part}.json", doc)

def load_data_file(name):
    """A data file from the working folder, or else from the repo on GitHub (if REPO is set)."""
    path = DATA / name
    if path.exists():
        return json.loads(path.read_text())
    if REPO:
        url = f"https://raw.githubusercontent.com/{REPO}/main/data/{name}"
        with urllib.request.urlopen(url, timeout=60) as r:
            text = r.read().decode()
        path.write_text(text)
        return json.loads(text)
    if IN_COLAB:
        from google.colab import files
        print(f"{name} isn't in {DATA}. Choose it in the upload box below (you can pick several of notebook 01's files at once).")
        for fname, content in files.upload().items():
            clean = re.sub(r" \(\d+\)(?=\.)", "", Path(fname).name)       # "field (1).json" -> "field.json"
            (DATA / clean).write_bytes(content)
        if path.exists():
            return json.loads(path.read_text())
    raise FileNotFoundError(f"{name} isn't in {DATA}. Run notebook 01 first (same Google account and DRIVE_FOLDER), "
                            "or set REPO so the file can be fetched from GitHub.")

def write_json(path, obj):
    path.write_text(json.dumps(obj, ensure_ascii=False, separators=(",", ":"), allow_nan=False))
    return path.stat().st_size / 1e6

GRID = load_data_file("grid.geojson")
FIELD_DOC = load_data_file("field.json")
FIELD = FIELD_DOC["field"]
SQUARES = [f["properties"] for f in GRID["features"]]
for sq in SQUARES:            # lower-left corner in metres, read from the EEA code, e.g. 1kmE4836N1990
    e, n = re.match(r"1kmE(\d+)N(\d+)", sq["id"]).groups()
    sq["x0"], sq["y0"] = int(e) * 1000, int(n) * 1000
FARM = [sq for sq in SQUARES if sq.get("farm")]
centre = shape(FIELD["outline"]).centroid                 # the field's data are worked out at the centre of its outline
FIELD_LON, FIELD_LAT = round(centre.x, 6), round(centre.y, 6)
FIELD_KEY = hashlib.md5(json.dumps(FIELD["outline"], sort_keys=True).encode()).hexdigest()[:8]   # changes if the outline is redrawn
FIELD_PT = ee.Geometry.Point([FIELD_LON, FIELD_LAT])
FIELD_GEOM = ee.Geometry(FIELD["outline"])
FIELD_AREA_M2 = FIELD["area_ha"] * 1e4

WC = ee.ImageCollection("ESA/WorldCover/v200").first().select("Map")
CLC = ee.Image("COPERNICUS/CORINE/V20/100m/2018").select("landcover")
RAINFED = WC.eq(40).And(CLC.eq(211))          # the project's rain-fed arable mask (10 m)

def square(sq):
    return ee.Geometry.Rectangle([sq["x0"], sq["y0"], sq["x0"] + CELL, sq["y0"] + CELL], "EPSG:3035", False)

def squares_fc(batch):
    return ee.FeatureCollection([ee.Feature(square(sq), {"id": sq["id"]}) for sq in batch])

print(f"{len(SQUARES):,} grid squares ({len(FARM):,} farmland); demo field in {FIELD['square']} "
      f"({FIELD['area_ha']} ha, outline {FIELD['outline_source'].split(':')[0]})")

## 1. Settings for greenness
- MODIS (NASA Terra, 250 m, every 16 days since 2000) gives the long record for every square. Only pixels that are mostly rain-fed arable land and have good-quality values are used.
- HLS (NASA's harmonised Landsat and Sentinel-2, 30 m) and Sentinel-2 (10 m) give sharper views of the demo field. A view counts only if at least 80% of the field is cloud-free in it.
- Nearby fields: the rain-fed arable land within 5 km (MODIS) or 3 km (HLS, Sentinel-2) of the demo field. That's the comparison for spotting a field doing worse than its neighbours.

In [ ]:
NOTEBOOK = "notebooks/02_greenness.ipynb"
TODAY = dt.date.today()
LAST_SEASON = TODAY.year if TODAY.month >= 9 else TODAY.year - 1     # last season that has finished
SEASONS = list(range(2001, LAST_SEASON + 1))
GREEN, BARE = 0.5, 0.3           # NDVI thresholds for "green crop" and "bare or nearly bare" months
MIN_CLEAR = 0.8                  # share of the field that must be cloud-free for a 30 m / 10 m view to count

MODIS = ee.ImageCollection("MODIS/061/MOD13Q1")
MPROJ = MODIS.first().select("NDVI").projection()
MASK250 = (RAINFED.unmask(0).reproject(WC.projection())
           .reduceResolution(ee.Reducer.mean(), maxPixels=1024).reproject(MPROJ).gte(0.5))  # MODIS pixels >= 50% rain-fed arable
RF = RAINFED.selfMask()
ZONE = int((FIELD_LON + 180) // 6) + 1          # UTM zone of the field (33 for the Bradano): satellite tiles are named after it
NBR5 = FIELD_PT.buffer(5000)
NBR3 = FIELD_PT.buffer(3000)

SOURCES = {
    "modis_ndvi": {"name": "MODIS Terra Vegetation Indices 16-day 250 m (MOD13Q1 v6.1)", "ee_id": "MODIS/061/MOD13Q1", "res_m": 250,
                   "licence": "NASA data: no restrictions on use",
                   "cite": "Didan, K. (2021). MOD13Q1 MODIS/Terra Vegetation Indices 16-Day L3 Global 250m SIN Grid V061. NASA LP DAAC. doi:10.5067/MODIS/MOD13Q1.061"},
    "hls": {"name": "Harmonized Landsat Sentinel-2 (HLSL30 and HLSS30 v2.0)", "ee_id": "NASA/HLS/HLSL30/v002, NASA/HLS/HLSS30/v002", "res_m": 30,
            "licence": "NASA data: no restrictions on use",
            "cite": "Masek, J. et al. (2021). HLS Operational Land Surface Reflectance and TOA Brightness Daily Global 30m v2.0. NASA LP DAAC. doi:10.5067/HLS/HLSL30.002, doi:10.5067/HLS/HLSS30.002"},
    "sentinel2": {"name": "Sentinel-2 surface reflectance (harmonised) with Cloud Score+", "ee_id": "COPERNICUS/S2_SR_HARMONIZED, GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED", "res_m": 10,
                  "licence": "Copernicus Sentinel data: free, full and open; Cloud Score+ CC BY 4.0",
                  "cite": "Contains modified Copernicus Sentinel data (ESA). Pasquarella, V. J. et al. (2023). Cloud Score+. CVPR Workshops"},
    "worldcover": GRID["meta"]["sources"]["worldcover"],
    "corine": GRID["meta"]["sources"]["corine"],
}
print(f"Seasons {SEASONS[0]}-{SEASONS[-1]}; normal = {BASELINE[0]}-{BASELINE[1]}")

## 2. Season maths
The same rules for every square and for the field, so the numbers can be compared:
- short cloud gaps (up to 3 composites) are filled by joining the dots;
- **peak** = highest greenness between February and June (an earlier bump is weeds or a flush after autumn rain, not the durum crop);
- **green-up** and **dry-off** = when greenness rises above, and later falls below, halfway between the autumn low and the peak (days counted from 1 September);
- **bare months** = months with average greenness below 0.3.

In [ ]:
def season_bounds(season):
    return pd.Timestamp(season - 1, 9, 1), pd.Timestamp(season, 9, 1)

def season_index(dates):
    # Positions of each season's composites in a date list (worked out once, shared by every square)
    dates = pd.DatetimeIndex(dates)
    idx = {}
    for season in SEASONS:
        start, stop = season_bounds(season)
        sel = np.flatnonzero((dates >= start) & (dates < stop))
        if len(sel) == 0:
            continue
        d = dates[sel]
        idx[season] = {"sel": sel, "day": np.asarray((d - start).days),
                       "peak_win": np.asarray((d >= pd.Timestamp(season, 2, 1)) & (d < pd.Timestamp(season, 7, 1))),
                       "autumn": np.asarray(d < pd.Timestamp(season - 1, 12, 1)),
                       "rise_win": np.asarray(d >= pd.Timestamp(season - 1, 10, 1)), "month": np.asarray(d.month)}
    return idx

def season_metrics(values, sidx):
    # values: NDVI array matching the dates used for sidx (NaN = missing). Returns {season: {...}}
    v = pd.Series(values, dtype="float64").interpolate(limit=3, limit_area="inside").to_numpy()
    out = {}
    for season, ix in sidx.items():
        w = v[ix["sel"]]
        ok = np.isfinite(w)
        if ok.sum() < 15:
            continue
        pk, aut = ix["peak_win"] & ok, ix["autumn"] & ok
        if not pk.any() or not aut.any():
            continue
        ipk = np.flatnonzero(pk)[np.argmax(w[pk])]
        peak, base = w[ipk], w[aut].min()
        half = base + 0.5 * (peak - base)
        pos = np.arange(len(w))
        greenup = dryoff = np.nan
        if peak - base >= 0.1:                                   # a real crop cycle
            day = ix["day"].astype(float)
            def cross(i, j):
                """Day the line between composites i and j passes the half-way level (dates between composites)."""
                return day[i] + (half - w[i]) / (w[j] - w[i]) * (day[j] - day[i]) if w[j] != w[i] else day[j]
            rise = np.flatnonzero(ix["rise_win"] & ok & (pos <= ipk))
            below = rise[w[rise] < half]
            after_low = rise[rise > below[-1]] if len(below) else rise
            if len(after_low):
                greenup = cross(below[-1], after_low[0]) if len(below) else day[after_low[0]]
            fall = np.flatnonzero(ok & (pos > ipk))
            down = fall[w[fall] < half]
            if len(down):
                prev = np.flatnonzero(ok & (pos >= ipk) & (pos < down[0]))[-1]       # last composite still above half
                dryoff = cross(prev, down[0])
        bare = sum(1 for m in np.unique(ix["month"]) if (ok & (ix["month"] == m)).any()
                   and w[ok & (ix["month"] == m)].mean() < BARE)
        out[season] = {"peak": float(peak), "peak_day": int(ix["day"][ipk]), "greenup_day": greenup,
                       "dryoff_day": dryoff, "season_days": dryoff - greenup, "bare_months": bare}
    return out

def month_climate(df):
    # For each row of df (columns = dates): 12 x [low, typical, high] of monthly mean greenness over the normal years
    d = pd.DatetimeIndex(df.columns)
    monthly = df.T.groupby([d.year, d.month]).mean().T
    out = {sid: [] for sid in df.index}
    for k in range(1, 13):
        cols = [c for c in monthly.columns if c[1] == k and BASELINE[0] <= c[0] <= BASELINE[1]]
        block = monthly[cols]
        n = block.notna().sum(axis=1)
        q = block.quantile([0.1, 0.5, 0.9], axis=1).T
        for sid in df.index:
            if n[sid] >= 15:
                out[sid].append([num(q.loc[sid, 0.1], 2), num(q.loc[sid, 0.5], 2), num(q.loc[sid, 0.9], 2)])
            else:
                out[sid].append([None, None, None])
    return out

def typical(table, key, nd):
    vals = [v[key] for s, v in table.items() if BASELINE[0] <= s <= BASELINE[1]]
    return lth(vals, nd)

print("Season maths ready")

## 3. The demo field
MODIS at the field's pixel plus the 25th, 50th and 75th percentiles of nearby rain-fed fields; then every HLS and Sentinel-2 overpass, kept only when the field is at least 80% clear. Takes about 3–5 minutes; each year is saved as it finishes.

In [ ]:
def modis_field_rows(a, b):
    def one(img):
        ndvi = img.select("NDVI").multiply(0.0001)
        qa = img.select("SummaryQA")
        at = ndvi.addBands(qa).reduceRegion(ee.Reducer.first(), FIELD_PT, 250)
        nb = (ndvi.updateMask(qa.lte(1)).updateMask(MASK250)
              .reduceRegion(ee.Reducer.percentile([25, 50, 75]), NBR5, 250, maxPixels=1e8))
        return ee.Feature(None, {"t": img.get("system:time_start"), "ndvi": at.get("NDVI"), "qa": at.get("SummaryQA"),
                                 "p25": nb.get("NDVI_p25"), "p50": nb.get("NDVI_p50"), "p75": nb.get("NDVI_p75")})
    fc = MODIS.filterDate(a, b).map(one)
    return [f["properties"] for f in with_retries(lambda: fc.getInfo())["features"]]

def hls_prep(img, nir, sensor):
    clear = img.select("Fmask").bitwiseAnd(0b11110).eq(0)        # Fmask bits 1-4: cloud, near cloud, shadow, snow
    ndvi = img.normalizedDifference([nir, "B4"]).rename("ndvi").updateMask(clear)
    return ee.Image(ndvi.set({"system:time_start": img.get("system:time_start"), "sensor": sensor}))

def hls_rows(a, b):
    def col(cid):                 # tiles of the field's UTM zone only (tiles on the date line falsely match everywhere), skip near-total cloud
        return (ee.ImageCollection(cid).filterBounds(FIELD_GEOM).filterDate(a, b)
                .filter(ee.Filter.stringStartsWith("system:index", f"T{ZONE:02d}")).filter(ee.Filter.lt("CLOUD_COVERAGE", 90)))
    l30 = col("NASA/HLS/HLSL30/v002").map(lambda i: hls_prep(i, "B5", "L30"))
    s30 = col("NASA/HLS/HLSS30/v002").map(lambda i: hls_prep(i, "B8A", "S30"))
    def one(img):
        st = img.reduceRegion(ee.Reducer.mean().combine(ee.Reducer.count(), sharedInputs=True), FIELD_GEOM, 30, "EPSG:3035")
        nb = img.updateMask(RF).reduceRegion(ee.Reducer.percentile([25, 50, 75]), NBR3, 30, "EPSG:3035", maxPixels=1e8)
        return ee.Feature(None, {"t": img.get("system:time_start"), "sensor": img.get("sensor"),
                                 "ndvi": st.get("ndvi_mean"), "n": st.get("ndvi_count"),
                                 "p25": nb.get("ndvi_p25"), "p50": nb.get("ndvi_p50"), "p75": nb.get("ndvi_p75")})
    fc = l30.merge(s30).map(one)
    return [f["properties"] for f in with_retries(lambda: fc.getInfo())["features"]]

def s2_rows(a, b):
    csp = ee.ImageCollection("GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED")
    col = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(FIELD_GEOM).filterDate(a, b)
           .filter(ee.Filter.stringContains("system:index", f"_T{ZONE:02d}")).linkCollection(csp, ["cs_cdf"]))
    def one(img):
        clear = img.select("cs_cdf").gte(0.60)
        ix = (img.normalizedDifference(["B8", "B4"]).rename("ndvi")
              .addBands(img.normalizedDifference(["B8A", "B5"]).rename("ndre")).updateMask(clear))
        st = ix.reduceRegion(ee.Reducer.mean().combine(ee.Reducer.count(), sharedInputs=True), FIELD_GEOM, 10, "EPSG:3035")
        nb = ix.select("ndvi").updateMask(RF).reduceRegion(ee.Reducer.percentile([25, 50, 75]), NBR3, 20, "EPSG:3035", maxPixels=1e9)
        return ee.Feature(None, {"t": img.get("system:time_start"), "ndvi": st.get("ndvi_mean"), "ndre": st.get("ndre_mean"),
                                 "n": st.get("ndvi_count"), "p25": nb.get("ndvi_p25"), "p50": nb.get("ndvi_p50"),
                                 "p75": nb.get("ndvi_p75")})
    fc = col.map(one)
    return [f["properties"] for f in with_retries(lambda: fc.getInfo())["features"]]

def cached(name, fn):
    path = CACHE / f"02_{name}.csv"
    if path.exists():
        return pd.read_csv(path)
    df = pd.DataFrame(fn())
    df.to_csv(path, index=False)
    return df

end = (TODAY + dt.timedelta(days=1)).isoformat()
jobs = {f"modis_field_{FIELD_KEY}_{y}": (modis_field_rows, f"{y}-01-01" if y > 2000 else "2000-09-01", f"{y + 1}-01-01" if y < TODAY.year else end)
        for y in range(2000, TODAY.year + 1)}
def halves(y):                    # half-year requests keep each Earth Engine call small
    first, second = (f"{y}-01-01", f"{y}-07-01"), (f"{y}-07-01", f"{y + 1}-01-01")
    return [first, second] if y < TODAY.year else [first, (f"{y}-07-01", end)] if TODAY.month >= 7 else [(f"{y}-01-01", end)]
jobs.update({f"hls_field_{FIELD_KEY}_{a}": (hls_rows, a, b) for y in range(2016, TODAY.year + 1) for a, b in halves(y)})
jobs.update({f"s2_field_{FIELD_KEY}_{a}": (s2_rows, a, b) for y in range(2017, TODAY.year + 1) for a, b in halves(y)})

t0, parts = time.time(), {}
with cf.ThreadPoolExecutor(max_workers=6) as pool:
    futs = {pool.submit(cached, name, lambda f=f, a=a, b=b: f(a, b)): name for name, (f, a, b) in jobs.items()}
    for fut in cf.as_completed(futs):
        parts[futs[fut]] = fut.result()
print(f"Field data fetched in {time.time() - t0:.0f} s")

def combine(prefix):
    frames = [parts[k] for k in sorted(parts) if k.startswith(prefix) and len(parts[k])]
    df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=["t"])
    df["date"] = pd.to_datetime(df["t"], unit="ms").dt.normalize().astype("datetime64[ns]")
    return df.sort_values("date")

mf = combine("modis_field")
mf["date"] = mf["date"] + pd.Timedelta(days=8)                # middle of the 16-day composite
mf["ndvi_ok"] = mf["ndvi"].where(mf["qa"] <= 1)

def clear_views(df, extra=()):
    """Keep overpasses where at least MIN_CLEAR of the field is clear; one per day (and sensor)."""
    if df.empty or "n" not in df:
        return df.iloc[0:0]
    full = df["n"].max()                         # pixels in the field when a view covers all of it
    df = df[df["n"].fillna(0) >= MIN_CLEAR * full]
    return df.sort_values("n").groupby(["date", *extra], as_index=False).last().sort_values("date")

hls = clear_views(combine("hls_field"), ["sensor"])
s2 = clear_views(combine("s2_field"))
check(len(mf) > 500, f"MODIS at the field: {len(mf)} composites, {mf['ndvi_ok'].notna().sum()} good-quality")
hls_per_year = hls.groupby(hls["date"].dt.year).size().to_dict() if len(hls) else {}
s2_per_year = s2.groupby(s2["date"].dt.year).size().to_dict() if len(s2) else {}
check(len(hls) >= 200, f"HLS clear views of the field: {len(hls)} ({hls_per_year})", warn=True)
check(len(s2) >= 200, f"Sentinel-2 clear views of the field: {len(s2)} ({s2_per_year})", warn=True)

# MODIS and HLS should agree (NSW practice run: MODIS was 0.01 higher)
if len(hls):
    pair = pd.merge_asof(hls[["date", "ndvi"]].rename(columns={"ndvi": "hls"}),
                         mf[["date", "ndvi_ok"]].dropna(), on="date", direction="nearest", tolerance=pd.Timedelta(days=8)).dropna()
    gap = float((pair["ndvi_ok"] - pair["hls"]).median()) if len(pair) else float("nan")
    check(abs(gap) < 0.1, f"MODIS and HLS agree at the field: median difference {gap:+.3f} over {len(pair)} matching dates", warn=True)

In [ ]:
dates = pd.DatetimeIndex(mf["date"])
fidx = season_index(dates)
field_seasons = season_metrics(mf["ndvi_ok"].to_numpy(float), fidx)
nbr_seasons = season_metrics(mf["p50"].to_numpy(float), fidx)

def zscore(table, key, season):
    base = [v[key] for s, v in table.items() if BASELINE[0] <= s <= BASELINE[1] and np.isfinite(v[key])]
    if len(base) < 15 or season not in table or not np.isfinite(table[season][key]) or np.std(base) == 0:
        return None
    return num((table[season][key] - np.mean(base)) / np.std(base), 2)

gaps = {s: {"gap": field_seasons[s]["peak"] - nbr_seasons[s]["peak"]} for s in field_seasons if s in nbr_seasons}

season_rows = []
for s in SEASONS:
    if s not in field_seasons:
        continue
    v = field_seasons[s]
    season_rows.append({"season": s, "peak": num(v["peak"], 2), "peak_day": num(v["peak_day"], 0),
                        "greenup_day": num(v["greenup_day"], 0), "dryoff_day": num(v["dryoff_day"], 0),
                        "bare_months": v["bare_months"],
                        "peak_z": zscore(field_seasons, "peak", s), "dryoff_z": zscore(field_seasons, "dryoff_day", s),
                        "peak_vs_nearby": num(gaps[s]["gap"], 2) if s in gaps else None,
                        "peak_vs_nearby_z": zscore(gaps, "gap", s)})

def series(df, cols, nd=3):
    out = {"dates": df["date"].dt.strftime("%Y-%m-%d").tolist()}
    for c, key in cols.items():
        out[key] = [num(x, nd) for x in df[c]] if c in df else []
    return out

calendar = month_climate(pd.DataFrame([mf["ndvi_ok"].to_numpy(float)], index=["field"], columns=dates))["field"]
greenness = {
    "made_by": f"{NOTEBOOK} ({RUN_AT})", "field_key": FIELD_KEY,
    "about": "Greenness (NDVI): about 0.2 = bare soil, 0.8 = thick healthy crop. Days are counted from 1 September; "
             "a season is named by its harvest year. 'nearby' = rain-fed arable land within 5 km (MODIS) or 3 km (HLS, Sentinel-2).",
    "baseline": f"{BASELINE[0]}-{BASELINE[1]}", "thresholds": {"green": GREEN, "bare": BARE},
    "calendar": {"month_ndvi": calendar, "shape": "12 months (Jan-Dec) x [low, typical, high] = 10th percentile, median, "
                 f"90th percentile of monthly means over {BASELINE[0]}-{BASELINE[1]}", "source": "modis_ndvi", "res_m": 250},
    "typical": {**{k: typical(field_seasons, k, 2 if k == "peak" else 0)
                   for k in ("peak", "peak_day", "greenup_day", "dryoff_day", "season_days", "bare_months")},
                "peak_vs_nearby": typical(gaps, "gap", 2)},
    "nearby_gap_note": "peak_vs_nearby = this field's peak minus the peak of nearby rain-fed fields (MODIS). Judge a season by "
                       "peak_vs_nearby_z, how far it departs from the field's usual gap, not by the raw gap.",
    "dates_note": "green-up and dry-off are the days greenness passes half-way between its autumn low and its peak, found "
                  "between the 16-day MODIS composites (about a week either way); peak_day is the composite with the highest value.",
    "seasons": season_rows,
    "modis": {"source": "modis_ndvi", "res_m": 250, **series(mf, {"ndvi_ok": "ndvi", "p25": "nearby_p25", "p50": "nearby_p50", "p75": "nearby_p75"})},
    "hls": {"source": "hls", "res_m": 30, "sensor": hls["sensor"].tolist() if len(hls) else [],
            **series(hls, {"ndvi": "ndvi", "p25": "nearby_p25", "p50": "nearby_p50", "p75": "nearby_p75"})},
    "sentinel2": {"source": "sentinel2", "res_m": 10,
                  **series(s2, {"ndvi": "ndvi", "ndre": "ndre", "p25": "nearby_p25", "p50": "nearby_p50", "p75": "nearby_p75"})},
}
typ = greenness["typical"]
print("Typical season at the field:", {k: v[1] for k, v in typ.items()})
for s in (2024, 2026):
    r = next((x for x in season_rows if x["season"] == s), None)
    if r:
        print(f"Season {s}: peak {r['peak']} (z {r['peak_z']}), dry-off day {r['dryoff_day']} (z {r['dryoff_z']}), "
              f"vs nearby {r['peak_vs_nearby']:+} (usual {typ['peak_vs_nearby'][1]}, z {r['peak_vs_nearby_z']})"
              if r["peak_vs_nearby"] is not None else f"Season {s}: {r}")

## 4. Every farmland square
For each year, one request per batch of squares gives the mean greenness of each square's rain-fed arable pixels for every 16-day composite. About 5–10 minutes the first time; each year is saved as it finishes.

In [ ]:
def modis_year_image(a, b):
    def clean(img):
        return (img.select("NDVI").multiply(0.0001).updateMask(img.select("SummaryQA").lte(1)).updateMask(MASK250)
                .rename(ee.String("d").cat(img.date().format("YYYYMMdd"))))
    return MODIS.filterDate(a, b).map(clean).toBands()

def year_batch(args):
    a, b, batch = args
    out = modis_year_image(a, b).reduceRegions(collection=squares_fc(batch), reducer=ee.Reducer.mean(), tileScale=4)
    feats = with_retries(lambda: out.getInfo())["features"]
    rows = []
    for f in feats:
        p = f["properties"]
        row = {"id": p["id"]}
        for k, v in p.items():
            m = re.search(r"d(\d{8})$", k)
            if m:
                row[m.group(1)] = v
        rows.append(row)
    return rows

def fetch_year(y):
    path = CACHE / f"02_cells_{y}.csv"
    if path.exists():
        return pd.read_csv(path, index_col="id")
    a = f"{y}-01-01" if y > 2000 else "2000-09-01"
    b = f"{y + 1}-01-01" if y < TODAY.year else end
    batches = [(a, b, FARM[i:i + 850]) for i in range(0, len(FARM), 850)]
    rows = [r for part in map(year_batch, batches) for r in part]
    df = pd.DataFrame(rows).set_index("id")
    df.to_csv(path)
    return df

t0, frames = time.time(), {}
with cf.ThreadPoolExecutor(max_workers=8) as pool:
    futs = {pool.submit(fetch_year, y): y for y in range(2000, TODAY.year + 1)}
    for fut in cf.as_completed(futs):
        frames[futs[fut]] = fut.result()
        print(f"  {futs[fut]} done ({len(frames)}/{len(futs)}, {time.time() - t0:.0f} s)")

wide = pd.concat([frames[y] for y in sorted(frames)], axis=1)
wide.columns = (pd.to_datetime(wide.columns, format="%Y%m%d") + pd.Timedelta(days=8)).astype("datetime64[ns]")
wide = wide.loc[:, ~wide.columns.duplicated()].sort_index(axis=1).reindex([sq["id"] for sq in FARM])
print(f"{wide.shape[0]:,} squares x {wide.shape[1]} composites; {wide.notna().mean().mean():.0%} of values present")

In [ ]:
cdates = pd.DatetimeIndex(wide.columns)
cidx = season_index(cdates)
enough = wide.notna().sum(axis=1) >= 100
climate = month_climate(wide[enough])
cells_out, all_tables = {}, {}
t0 = time.time()
for sid, row in wide[enough].iterrows():
    table = season_metrics(row.to_numpy(float), cidx)
    all_tables[sid] = table
    cells_out[sid] = {
        "ndvi_month": climate[sid],
        "peak_ndvi": typical(table, "peak", 2),
        "peak_day": typical(table, "peak_day", 0),
        "greenup_day": typical(table, "greenup_day", 0),
        "dryoff_day": typical(table, "dryoff_day", 0),
        "season_days": typical(table, "season_days", 0),
        "bare_months": typical(table, "bare_months", 0),
        "peak_by_season": [num(table[s]["peak"], 2) if s in table else None for s in SEASONS],
        "dryoff_by_season": [num(table[s]["dryoff_day"], 0) if s in table else None for s in SEASONS],
    }
print(f"Season numbers for {len(cells_out):,} squares in {time.time() - t0:.0f} s")
check(len(cells_out) >= 0.9 * len(FARM), f"Greenness for {len(cells_out):,} of {len(FARM):,} farmland squares")

# Test 1: the calendar should look like a winter crop (green in late winter/spring, bare in late summer)
typ_month = np.nanmedian(np.array([[m[1] if m[1] is not None else np.nan for m in c["ndvi_month"]] for c in cells_out.values()]), axis=0)
peak_month, low_month = int(np.nanargmax(typ_month)) + 1, int(np.nanargmin(typ_month)) + 1
check(peak_month in (2, 3, 4, 5) and low_month in (6, 7, 8, 9, 10),
      f"Winter-crop calendar: typical greenness peaks in month {peak_month} and is lowest in month {low_month}")

# Test 2: known events. 2024 was a drought (large durum losses); 2026 followed a very wet winter
def area_peak(season):
    v = [t[season]["peak"] for t in all_tables.values() if season in t]
    return float(np.median(v)) if v else float("nan")
normal_peak = float(np.median([area_peak(s) for s in range(BASELINE[0], BASELINE[1] + 1)]))
p24, p26 = area_peak(2024), area_peak(2026)
check(p24 < normal_peak, f"Drought test: area peak greenness 2024 = {p24:.2f}, below the normal {normal_peak:.2f}", warn=True)
if np.isfinite(p26):
    check(p26 > p24, f"Wet-year test: area peak greenness 2026 = {p26:.2f}, above 2024", warn=True)

## 5. Write the files

In [ ]:
INDICATORS = {
    "ndvi_month": {"what": "typical greenness (NDVI) in each month", "shape": "12 months (Jan-Dec) x [low, typical, high]"},
    "peak_ndvi": {"what": "highest greenness of the season (February-June)", "shape": "[low, typical, high]"},
    "peak_day": {"what": "day of the peak, counted from 1 September", "shape": "[low, typical, high]"},
    "greenup_day": {"what": "day greenness rises past halfway to the peak, counted from 1 September (found between 16-day composites)", "shape": "[low, typical, high]"},
    "dryoff_day": {"what": "day greenness falls back below halfway, counted from 1 September (found between 16-day composites; early = crop finished early)", "shape": "[low, typical, high]"},
    "season_days": {"what": "days from green-up to dry-off", "shape": "[low, typical, high]"},
    "bare_months": {"what": f"months per season with average greenness below {BARE}", "shape": "[low, typical, high]"},
    "peak_by_season": {"what": "peak greenness of each season", "shape": "one value per season in meta.seasons"},
    "dryoff_by_season": {"what": "dry-off day of each season", "shape": "one value per season in meta.seasons"},
}
for v in INDICATORS.values():
    v.update({"source": "modis_ndvi", "res_m": 250})
cells_doc = {"meta": {"project": "Field Shift (NASA Space Apps Challenge 2026)", "made_by": f"{NOTEBOOK} ({RUN_AT})",
                      "keys": "grid square ids from grid.geojson (farmland squares only)",
                      "low_typical_high": f"10th percentile, median and 90th percentile of the seasons {BASELINE[0]}-{BASELINE[1]}",
                      "baseline": f"{BASELINE[0]}-{BASELINE[1]}", "seasons": SEASONS,
                      "season": "named by harvest year: season 2024 = September 2023 to August 2024",
                      "pixels": "MODIS pixels that are at least half rain-fed arable land (WorldCover cropland and CORINE 211), good quality only",
                      "thresholds": {"green": GREEN, "bare": BARE}, "indicators": INDICATORS, "sources": SOURCES},
             "cells": cells_out}

OUTPUTS = ["field_greenness.json", "cells_greenness.json"]
check(write_field_section("greenness", {"greenness": greenness}, SOURCES) < 5, "field_greenness.json written")
mb = write_json(DATA / "cells_greenness.json", cells_doc)
check(mb < 5, f"cells_greenness.json written: {mb:.2f} MB")

## 6. One chart to explain to a farmer
Each grey line is one season at the demo field; the band is the normal range. Red is the 2024 drought season, blue is 2026.

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(10, 4.5))
s = pd.Series(mf["ndvi_ok"].to_numpy(float), index=dates).interpolate(limit=3, limit_area="inside")
curves = {}
for season in SEASONS:
    start, stop = season_bounds(season)
    w = s[(s.index >= start) & (s.index < stop)]
    curves[season] = pd.Series(w.values, index=(w.index - start).days)
    ax.plot(curves[season].index, curves[season].values, color="#bbbbbb", lw=0.8)
base = pd.concat([curves[y] for y in curves if BASELINE[0] <= y <= BASELINE[1]], axis=1)
bins = base.index // 16
binned = base.groupby(bins).median()
x = pd.Series(base.index, index=base.index).groupby(bins).mean()
ax.fill_between(x.values, binned.quantile(0.1, axis=1).values, binned.quantile(0.9, axis=1).values, color="#7fb77e", alpha=0.35,
                label=f"Normal range ({BASELINE[0]}-{BASELINE[1]})")
for season, colour in ((2024, "#c0392b"), (2026, "#1f6fb2")):
    if season in curves:
        ax.plot(curves[season].index, curves[season].values, color=colour, lw=2.5, label=f"Season {season}")
ticks = pd.date_range("2001-09-01", "2002-08-01", freq="MS")
ax.set_xticks([(t - ticks[0]).days for t in ticks], [t.strftime("%b") for t in ticks])
ax.set_ylabel("Greenness (NDVI)")
ax.set_title("Demo field, Genzano di Lucania: greenness through each season (MODIS, 250 m)", loc="left")
ax.legend(frameon=False, loc="upper left")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(DATA.parent / "greenness_field.png", dpi=150)
plt.show()

## 7. Checks, downloads and what to paste back

In [ ]:
print("\n".join(CHECKS))
bad = [c for c in CHECKS if c.startswith("❌")]
print("\nAll essential checks passed." if not bad else f"\n{len(bad)} check(s) failed: paste the lines above into the chat.")
if IN_COLAB:
    from google.colab import files
    for name in OUTPUTS:
        files.download(str(DATA / name))
    print("Downloading", ", ".join(OUTPUTS), "- upload them to the repo's data folder (replace the old ones).")

In [ ]:
print("=== Paste this back to Claude ===")
print(f"02_greenness run {RUN_AT}; seasons {SEASONS[0]}-{SEASONS[-1]}")
print("field typical:", json.dumps(greenness["typical"]))
print("field calendar (typical NDVI Jan-Dec):", [m[1] for m in calendar])
print("field seasons 2017-:", json.dumps([r for r in season_rows if r["season"] >= 2017]))
print(f"clear views per year: HLS {hls_per_year}; Sentinel-2 {s2_per_year}")
print(f"area: {len(cells_out)} squares; typical NDVI by month {np.round(typ_month, 2).tolist()}")
print(f"area peak: normal {normal_peak:.2f}; 2024 {p24:.2f}; 2026 {p26:.2f}")
print("\n".join(CHECKS))